# Phase 3 TFIM n × h scaling diagnostic

Bounded overnight-safe diagnostic for whether TFIM forecast dispersion/tail skill improves with qubit count and transverse-field tuning.


In [ ]:
from pathlib import Path
import os, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 240)

def find_repo_root():
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if p.name == 'qpitome-qrc-volatility' and (p/'scripts').exists():
            return p
    raise RuntimeError('Open from inside qpitome-qrc-volatility')

ROOT = find_repo_root(); os.chdir(ROOT)
TABLES = ROOT/'results'/'tables'; FIGURES = ROOT/'results'/'figures'
TABLES.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)
print(ROOT)


## Run scaling grid

In [ ]:
cmd = [
    sys.executable, 'scripts/run_phase3_tfim_n_h_scaling.py',
    '--n-qubits', '4', '6', '8', '10', '12',
    '--h-grid', '0.2', '0.5', '1.0',
    '--seeds', '0', '1', '2',
    '--max-train', '1600', '--max-val', '500', '--max-test', '700',
    '--layers', '4', '--ridge-alpha', '3000',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


## Load outputs

In [ ]:
metrics = pd.read_csv(TABLES/'phase3_tfim_n_h_scaling_metrics.csv')
test = pd.read_csv(TABLES/'phase3_tfim_n_h_scaling_test_summary.csv')
agg = pd.read_csv(TABLES/'phase3_tfim_n_h_scaling_test_agg.csv')

display(test.sort_values('qlike').head(15))
display(test.sort_values('q95_f1', ascending=False).head(15))
display(agg)


## Figures

In [ ]:
def savefig(path):
    plt.tight_layout(); plt.savefig(path, dpi=180, bbox_inches='tight'); print('Saved:', path); plt.show()

for metric in ['qlike_mean','corr_mean','pred_std_mean','q90_f1_mean','q95_f1_mean','effective_rank_mean']:
    fig, ax = plt.subplots(figsize=(8,5))
    for h, g in agg.groupby('h'):
        g = g.sort_values('n_qubits')
        ax.plot(g['n_qubits'], g[metric], marker='o', label=f'h={h}')
    ax.set_xlabel('n qubits'); ax.set_ylabel(metric); ax.set_title(f'TFIM n × h scaling: {metric}')
    ax.legend(); savefig(FIGURES/f'phase3_tfim_n_h_scaling_{metric}.png')

fig, ax = plt.subplots(figsize=(7,5))
tfim = test[test['n_qubits'] > 0]
ax.scatter(tfim['qlike'], tfim['q95_f1'])
for _, r in tfim.iterrows():
    if r['q95_f1'] >= tfim['q95_f1'].quantile(.85) or r['qlike'] <= tfim['qlike'].quantile(.15):
        ax.annotate(f"{int(r['n_qubits'])}q h={r['h']} s={int(r['seed'])}", (r['qlike'], r['q95_f1']), xytext=(5,5), textcoords='offset points', fontsize=7)
ax.set_xlabel('repo-style QLIKE (lower better)'); ax.set_ylabel('q95 F1')
ax.set_title('TFIM scaling: QLIKE vs q95 F1')
savefig(FIGURES/'phase3_tfim_n_h_scaling_qlike_vs_q95.png')


## Decision summary

In [ ]:
har = test[test['model'].eq('HAR_memory_ridge')].iloc[0]
tfim = test[test['n_qubits'] > 0]
best_qlike = tfim.sort_values('qlike').iloc[0]
best_tail = tfim.sort_values('q95_f1', ascending=False).iloc[0]
print('HAR qlike:', har['qlike'], 'HAR q95 F1:', har['q95_f1'], 'HAR pred_std:', har['pred_std'])
print('\nBest TFIM by qlike:')
print(best_qlike[['model','n_qubits','h','seed','qlike','corr','pred_std','q90_f1','q95_f1','effective_rank']].to_string())
print('\nBest TFIM by q95 F1:')
print(best_tail[['model','n_qubits','h','seed','qlike','corr','pred_std','q90_f1','q95_f1','effective_rank']].to_string())
